In [ ]:
# Code Cell: Imports and display settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"

# Colab-friendly fallback for the uploaded file in this environment
import os
if not os.path.exists(DATA_PATH) and os.path.exists("/mnt/data/StudentsPerformance.csv"):
    DATA_PATH = "/mnt/data/StudentsPerformance.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


# Student Performance Analysis — Education Domain

**Dataset:** `StudentsPerformance.csv`  
**Approach:** Senior Data Analyst review for an international-school education context.

### Analysis rules
- Use **only the information contained in the dataset**.
- **Do not remove any records**.
- Validate missing values, duplicates, data types, categorical values, numeric ranges, and outliers.
- Perform univariate, bivariate, and multivariate EDA.
- Answer business/education questions only when the data supports the answer.
- For unsupported questions (for example, sales trends without dates or sales fields), explicitly state that the answer cannot be determined from this dataset.


## 1. Data Intake Overview

The dataset contains **1,000 student records and 8 columns**:
- 5 categorical variables: gender, race/ethnicity, parental education, lunch, and test preparation.
- 3 numerical variables: math, reading, and writing scores.
- There is no student ID, date, sales amount, product, store, region, or geographic field.


In [ ]:
# Code Cell: Structure and data types
df.info()

print("\nShape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))


In [ ]:
# Code Cell: Numerical and categorical summaries
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

print("\nNumerical statistics:")
display(df[numeric_cols].describe().T)

print("\nCategorical statistics:")
display(df[categorical_cols].describe().T)


### Intake insights

- The dataset has **1,000 rows and 8 columns**.
- The three score fields are stored as integers.
- The five demographic/context fields are stored as text categories.
- Math has the lowest mean score (**66.09**), followed by writing (**68.05**) and reading (**69.17**).
- Average student performance across the three subjects is **67.77**.


## 2. Data Quality Validation

The validation checks cover:
1. Missing/null values.
2. Exact duplicate rows.
3. Unexpected categorical values.
4. Invalid score ranges.
5. Data types.
6. Outliers using the IQR rule.

Outliers are **flagged, not removed**, because the requirement is to preserve all data.


In [ ]:
# Code Cell: Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})
display(missing_report)


### Missing-value insight

**There are no missing values in any column.**  
Therefore, no imputation is required for the current dataset.


In [ ]:
# Code Cell: Duplicate validation
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

# Also inspect duplicate frequency without deleting anything
duplicate_rows = df[df.duplicated(keep=False)].sort_values(list(df.columns))
display(duplicate_rows.head(20))


### Duplicate insight

**There are no exact duplicate rows.**  
No records are removed.


In [ ]:
# Code Cell: Validate categorical values
expected_categories = {
    "gender": {"female", "male"},
    "race/ethnicity": {"group A", "group B", "group C", "group D", "group E"},
    "parental level of education": {
        "some high school", "high school", "some college",
        "associate's degree", "bachelor's degree", "master's degree"
    },
    "lunch": {"standard", "free/reduced"},
    "test preparation course": {"none", "completed"}
}

categorical_validation = []
for col, expected in expected_categories.items():
    observed = set(df[col].dropna().unique())
    invalid = sorted(observed - expected)
    categorical_validation.append({
        "column": col,
        "observed_unique": len(observed),
        "invalid_values": invalid,
        "status": "PASS" if not invalid else "CHECK"
    })

display(pd.DataFrame(categorical_validation))


In [ ]:
# Code Cell: Validate numerical score ranges
score_cols = ["math score", "reading score", "writing score"]

range_validation = []
for col in score_cols:
    invalid_mask = ~df[col].between(0, 100, inclusive="both")
    range_validation.append({
        "column": col,
        "min": df[col].min(),
        "max": df[col].max(),
        "invalid_count": int(invalid_mask.sum()),
        "status": "PASS" if invalid_mask.sum() == 0 else "CHECK"
    })

display(pd.DataFrame(range_validation))


In [ ]:
# Code Cell: IQR outlier detection — flag only, never delete
outlier_report = []

for col in score_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)

    outlier_report.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_pct": mask.mean() * 100
    })

outlier_report = pd.DataFrame(outlier_report)
display(outlier_report)


### Outlier insight

The IQR rule flags:
- **8 math scores**
- **6 reading scores**
- **5 writing scores**

These observations are retained. In an education setting, unusually low or high scores can be genuine student outcomes, so they should be investigated rather than automatically deleted.


## 3. Data Cleaning — Preserve All Records

Because the current data has no nulls, no duplicate rows, no invalid categories, and no scores outside 0–100, **no corrective changes are required**.

The following cleaning framework is included for reproducibility. If future data contains missing values, it fills them without deleting rows:
- Numerical scores → median.
- Categorical variables → mode.
- Invalid score values → treated as missing and then imputed.
- Invalid categorical values → treated as missing and then imputed.

This is a fallback recommendation, not a change applied to the current clean dataset.


In [ ]:
# Code Cell: Non-destructive cleaning function
clean_df = df.copy()

# Clean categorical whitespace without changing the row count
for col in categorical_cols:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Convert score columns to numeric; invalid text becomes NaN
for col in score_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Values outside the valid score range are invalid.
for col in score_cols:
    clean_df.loc[~clean_df[col].between(0, 100), col] = np.nan

# Impute only if needed; current dataset has no missing values.
for col in score_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in categorical_cols:
    mode_value = clean_df[col].mode(dropna=True)
    if len(mode_value) > 0:
        clean_df[col] = clean_df[col].fillna(mode_value.iloc[0])

print("Original shape:", df.shape)
print("Cleaned shape :", clean_df.shape)
print("Rows removed  :", len(df) - len(clean_df))
print("\nRemaining nulls:")
display(clean_df.isna().sum().to_frame("null_count"))


## 4. Feature Engineering

To analyze overall student performance, an **average score** is calculated from the three available subjects.

For the pass-rate analysis only, a transparent analytical threshold of **50/100 per subject** is used. The dataset itself does not provide an official school pass/fail rule, so this threshold should be treated as an analysis assumption rather than an official grading policy.


In [ ]:
# Code Cell: Derived metrics
analysis_df = clean_df.copy()

analysis_df["average_score"] = analysis_df[score_cols].mean(axis=1)
analysis_df["total_score"] = analysis_df[score_cols].sum(axis=1)

# Analytical threshold only: 50/100 in every subject
analysis_df["passed_all_50"] = (analysis_df[score_cols] >= 50).all(axis=1)
analysis_df["failed_any_50"] = (analysis_df[score_cols] < 50).any(axis=1)

display(analysis_df[score_cols + ["average_score", "total_score", "passed_all_50"]].head())


## 5. Univariate EDA

### Overall score distribution
The overall average score is **67.77/100**. Reading has the highest mean (**69.17**), while math has the lowest (**66.09**).

The score distributions are broad enough to show meaningful differences between students.


In [ ]:
# Code Cell: Univariate distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, score_cols):
    ax.hist(analysis_df[col], bins=20)
    ax.set_title(f"{col.title()} Distribution")
    ax.set_xlabel("Score")
    ax.set_ylabel("Students")

plt.tight_layout()
plt.show()


In [ ]:
# Code Cell: Categorical distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, categorical_cols):
    counts = analysis_df[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(f"{col.title()} Distribution")
    ax.tick_params(axis="x", rotation=30)

axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 6. Important Questions — Answered Only From the Data

### Q1. Which subject has the weakest average performance?
**Math**, with an average of **66.09**, is the lowest of the three subjects.

### Q2. Which subject has the strongest average performance?
**Reading**, with an average of **69.17**.

### Q3. Is performance different by gender?
Yes. Females have a higher overall average (**69.57**) than males (**65.84**).  
However, males have a higher math average (**68.73 vs 63.63**), while females have higher reading and writing averages.

### Q4. Does lunch status show a performance difference?
Yes. Students with standard lunch average **70.84**, compared with **62.20** for free/reduced lunch.

### Q5. Is test preparation associated with higher scores?
Within this dataset, students who completed the preparation course have an average of **72.67**, versus **65.04** for students who did not.

This is an observed association in the dataset, not proof that the course itself caused the improvement.

### Q6. Which race/ethnicity group has the highest average?
Group E has the highest average (**72.75**), while Group A has the lowest (**62.99**).

### Q7. Does parental education differ with student performance?
Yes. The highest average is for students whose parents have a master's degree (**73.60**), while the lowest is for students whose parents have only high school education (**63.10**).

### Q8. How many students pass all three subjects using the 50/100 analytical threshold?
**81.2%** pass all three subjects; **18.8%** have at least one score below 50.


## 7. Bivariate EDA

The most important pairwise comparisons are:
- Score vs gender.
- Score vs lunch.
- Score vs test preparation.
- Score vs race/ethnicity.
- Score vs parental education.


In [ ]:
# Code Cell: Average performance by key categorical variables
group_cols = [
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course"
]

for col in group_cols:
    summary = (
        analysis_df.groupby(col)
        .agg(
            students=("average_score", "size"),
            average_score=("average_score", "mean"),
            pass_rate=("passed_all_50", "mean")
        )
        .sort_values("average_score", ascending=False)
    )
    summary["pass_rate"] *= 100
    print(f"\n--- {col} ---")
    display(summary.round(2))


In [ ]:
# Code Cell: Boxplots for average score
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, group_cols):
    categories = list(analysis_df[col].dropna().unique())
    data = [analysis_df.loc[analysis_df[col] == cat, "average_score"] for cat in categories]
    ax.boxplot(data, labels=categories)
    ax.set_title(f"Average Score by {col.title()}")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("Average Score")

axes[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Code Cell: Subject means by gender, lunch, and preparation
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, col in zip(axes, ["gender", "lunch", "test preparation course"]):
    plot_df = analysis_df.groupby(col)[score_cols].mean()
    x = np.arange(len(plot_df.index))
    width = 0.24

    for i, subject in enumerate(score_cols):
        ax.bar(x + (i - 1) * width, plot_df[subject].values, width=width, label=subject)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=20)
    ax.set_title(f"Subject Performance by {col.title()}")
    ax.set_ylabel("Mean Score")
    ax.set_ylim(0, 100)
    ax.legend()

plt.tight_layout()
plt.show()


## 8. Multivariate EDA

The strongest multivariate questions are whether contextual variables work together and whether the same pattern is visible across subgroups.

The analysis below compares:
- Preparation × lunch.
- Gender × preparation.
- Race/ethnicity × preparation.
- All three subjects together.


In [ ]:
# Code Cell: Preparation × Lunch
prep_lunch = (
    analysis_df.groupby(["test preparation course", "lunch"])
    .agg(
        students=("average_score", "size"),
        average_score=("average_score", "mean"),
        pass_rate=("passed_all_50", "mean")
    )
    .reset_index()
)
prep_lunch["pass_rate"] *= 100
display(prep_lunch.round(2))


In [ ]:
# Code Cell: Heatmap — average score by preparation and lunch
pivot = analysis_df.pivot_table(
    index="test preparation course",
    columns="lunch",
    values="average_score",
    aggfunc="mean"
)

plt.figure(figsize=(8, 5))
plt.imshow(pivot.values, aspect="auto")
plt.colorbar(label="Average Score")
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.title("Average Score: Test Preparation × Lunch")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        plt.text(j, i, f"{pivot.iloc[i, j]:.2f}", ha="center", va="center")

plt.xlabel("Lunch")
plt.ylabel("Test Preparation")
plt.show()


In [ ]:
# Code Cell: Gender × Preparation
gender_prep = (
    analysis_df.groupby(["gender", "test preparation course"])["average_score"]
    .agg(["mean", "count"])
    .reset_index()
)
display(gender_prep.round(2))

genders = gender_prep["gender"].unique()
preps = gender_prep["test preparation course"].unique()
x = np.arange(len(genders))
width = 0.35

plt.figure(figsize=(9, 5))
for i, prep in enumerate(preps):
    vals = [
        gender_prep.loc[
            (gender_prep["gender"] == g) &
            (gender_prep["test preparation course"] == prep), "mean"
        ].iloc[0]
        for g in genders
    ]
    plt.bar(x + (i - 0.5) * width, vals, width=width, label=prep)

plt.xticks(x, genders)
plt.ylim(0, 100)
plt.title("Average Score: Gender × Test Preparation")
plt.ylabel("Average Score")
plt.legend()
plt.show()


In [ ]:
# Code Cell: Correlation among subjects
corr = analysis_df[score_cols].corr()

plt.figure(figsize=(7, 5))
plt.imshow(corr.values, vmin=0, vmax=1, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(score_cols)), score_cols, rotation=20)
plt.yticks(range(len(score_cols)), score_cols)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        plt.text(j, i, f"{corr.iloc[i, j]:.3f}", ha="center", va="center")

plt.title("Correlation Between Subject Scores")
plt.show()

display(corr.round(3))


### Multivariate insights

- Reading and writing are very strongly related (**r ≈ 0.955**).
- Math is also strongly related to reading (**r ≈ 0.818**) and writing (**r ≈ 0.803**).
- Preparation and lunch show a large combined difference: students with **completed preparation + standard lunch** have the highest subgroup average (**75.51**), while **no preparation + free/reduced lunch** is the lowest (**58.95**).
- Preparation is associated with higher average performance in every race/ethnicity group in this dataset.


In [ ]:
# Code Cell: Preparation effect within each race/ethnicity group
race_prep = (
    analysis_df.groupby(["race/ethnicity", "test preparation course"])["average_score"]
    .mean()
    .unstack()
)

race_prep["difference_completed_minus_none"] = race_prep["completed"] - race_prep["none"]
display(race_prep.round(2))


## 9. Performance Focus — Where Is Performance Dropping?

This dataset has **no time/date variable**, so a true performance trend over time cannot be calculated.

What can be identified is **where performance is lower across available categories**:
- Free/reduced lunch group: **62.20** average.
- No preparation course: **65.04** average.
- Group A: **62.99** average.
- Parental education = high school: **63.10** average.
- Math: **66.09**, the weakest subject overall.

### When is performance dropping?
**Cannot be determined from this dataset.** There is no date, month, academic year, term, cohort, or historical-period column.

### Where is performance lower?
Across the available categorical dimensions, the lowest observed group averages are concentrated around free/reduced lunch, no preparation, Group A, and high-school parental education.

### Why is performance lower?
The dataset can show **associations**, but it cannot establish causality. It does not contain attendance, teaching quality, socioeconomic detail beyond lunch status, study hours, prior grades, school/branch, teacher, or intervention data.


In [ ]:
# Code Cell: Rank the available segments by average performance
segment_results = []

for col in group_cols:
    tmp = (
        analysis_df.groupby(col)
        .agg(
            students=("average_score", "size"),
            average_score=("average_score", "mean"),
            pass_rate=("passed_all_50", "mean")
        )
        .reset_index()
        .rename(columns={col: "segment"})
    )
    tmp["dimension"] = col
    segment_results.append(tmp)

segment_results = pd.concat(segment_results, ignore_index=True)
segment_results["pass_rate"] *= 100

display(
    segment_results
    .sort_values(["dimension", "average_score"])
    .round(2)
)


## 10. Sales / Superstore Requirement

### Data limitation — important

The requested **sales increasing/dropping** analysis and **Superstore performance** analysis cannot be performed from this file.

The provided dataset is `StudentsPerformance.csv` and contains only student demographic/context fields and academic scores. It has **no sales, revenue, quantity, profit, product, category, customer, order date, store, region, or transaction fields**.

Therefore:
- **Sales increasing or dropping:** cannot be determined.
- **Why sales changed:** cannot be determined.
- **When sales changed:** cannot be determined.
- **Where sales changed:** cannot be determined.
- **Superstore performance:** cannot be calculated.

Creating sales conclusions from this dataset would violate the requirement to answer from data only.


## 11. Key Visual Summary

The next visuals highlight the most actionable patterns supported by the data:
1. Overall subject performance.
2. Average score by lunch.
3. Average score by preparation.
4. Average score by parental education.
5. Average score by race/ethnicity.
6. Pass rate by key factors.


In [ ]:
# Code Cell: Executive visual — subject means
subject_means = analysis_df[score_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(subject_means.index, subject_means.values)
plt.ylim(0, 100)
plt.ylabel("Mean Score")
plt.xlabel("Subject")
plt.title("Overall Mean Score by Subject")
plt.show()


In [ ]:
# Code Cell: Executive visual — strongest contextual drivers
context_means = {
    "Standard lunch": analysis_df.loc[analysis_df["lunch"] == "standard", "average_score"].mean(),
    "Free/reduced lunch": analysis_df.loc[analysis_df["lunch"] == "free/reduced", "average_score"].mean(),
    "Preparation completed": analysis_df.loc[analysis_df["test preparation course"] == "completed", "average_score"].mean(),
    "Preparation none": analysis_df.loc[analysis_df["test preparation course"] == "none", "average_score"].mean(),
}

context_plot = pd.Series(context_means).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(context_plot.index, context_plot.values)
plt.xlim(0, 100)
plt.xlabel("Average Score")
plt.ylabel("")
plt.title("Average Performance Across Key Context Groups")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Code Cell: Executive visual — pass rate
pass_rates = {
    "Overall": analysis_df["passed_all_50"].mean() * 100,
    "Female": analysis_df.loc[analysis_df["gender"] == "female", "passed_all_50"].mean() * 100,
    "Male": analysis_df.loc[analysis_df["gender"] == "male", "passed_all_50"].mean() * 100,
    "Standard lunch": analysis_df.loc[analysis_df["lunch"] == "standard", "passed_all_50"].mean() * 100,
    "Free/reduced lunch": analysis_df.loc[analysis_df["lunch"] == "free/reduced", "passed_all_50"].mean() * 100,
    "Prep completed": analysis_df.loc[analysis_df["test preparation course"] == "completed", "passed_all_50"].mean() * 100,
    "Prep none": analysis_df.loc[analysis_df["test preparation course"] == "none", "passed_all_50"].mean() * 100,
}

pass_plot = pd.Series(pass_rates).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(pass_plot.index, pass_plot.values)
plt.xlim(0, 100)
plt.xlabel("Pass Rate (%)")
plt.ylabel("")
plt.title("Pass Rate Using the 50/100 Analytical Threshold")
plt.gca().invert_yaxis()
plt.show()


## 12. Final Insights & Recommendations

### Data quality
- **1,000 records** were analyzed.
- **0 missing values**.
- **0 exact duplicates**.
- All observed categorical values match the expected categories.
- All observed scores are within the valid **0–100** range.
- IQR flags a small number of extreme scores, but they are retained.

### Academic performance
- Overall mean score: **67.77**.
- Reading is strongest: **69.17**.
- Math is weakest: **66.09**.
- **81.2%** of students meet the 50/100 threshold in all three subjects.

### Main segment differences
- Standard lunch: **70.84** average vs free/reduced: **62.20**.
- Preparation completed: **72.67** vs none: **65.04**.
- Group E: **72.75**, highest race/ethnicity group average.
- Master's-degree parental education: **73.60**, highest parental-education segment.
- Female overall average: **69.57** vs male: **65.84**.

### Recommendations supported by the data
1. **Prioritize math support**, because it has the lowest subject average.
2. **Increase access to or participation in test preparation**, because the completed-preparation group has materially higher observed scores.
3. **Prioritize support for lower-performing segments**, especially free/reduced lunch and no-preparation groups.
4. **Investigate flagged outliers individually**, rather than deleting them.
5. For future school analytics, collect **time/term, attendance, student ID, school/branch, teacher, intervention, and prior-performance fields**. These would allow trend, retention, cohort, and causal-investigation analyses.
6. Do not use the current dataset to make sales or Superstore claims; those require a separate transactional dataset.

### Final conclusion

The strongest pattern in the available data is that student performance differs substantially across **test preparation, lunch status, parental education, race/ethnicity, and gender**, while the subject-level relationship between reading and writing is particularly strong. These are **associations observed in the dataset**, not causal conclusions.


## 13. Reproducibility Note

No records were intentionally deleted during the analysis. Outliers were only flagged. The cleaning process is non-destructive and is designed to preserve row count.

**Source file:** `StudentsPerformance.csv`
